In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import climate_indices
from climate_indices import indices

df = pd.read_parquet('../data/processed/climate_mensal.parquet')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
precipitacao = df.loc[(slice(None), slice(None)), 'precipitacao_mm']
precipitacao.index = precipitacao.index.get_level_values('datetime')

precipitacao_media_anual = precipitacao.resample('YE').mean()
precipitacao_media_anual.index = precipitacao_media_anual.index.year

media_historica = precipitacao_media_anual.mean()
threshold = media_historica * 0.75

# Lógica de cores: vermelho para anos abaixo do limiar, azul para os demais
cores = ['red' if v < threshold else 'steelblue' for v in precipitacao_media_anual.values]

# Configuração e plotagem do gráfico
fig, ax = plt.subplots(figsize=(14, 4))
precipitacao_media_anual.plot(kind='bar', color=cores, ax=ax)

# Ajustes visuais (títulos e eixos)
ax.set_title('Precipitação Média Mensal por Ano — Todas as Estações')
ax.set_ylabel('Precipitação (mm)')
ax.set_xlabel('Ano')
ax.tick_params(axis='x', rotation=0)

# Linhas de referência
ax.axhline(y=media_historica, color='black', linestyle='--', linewidth=1, label='Média Histórica')
ax.axhline(y=threshold, color='red', linestyle=':', linewidth=1, label='Limiar de Alerta (75%)')

ax.legend()
plt.tight_layout()
plt.show()

A dinâmica da seca no Ceará é complexa e exige uma perspectiva de longo prazo. Devido à forte sazonalidade da região — onde as chuvas se concentram em uma "quadra chuvosa" em vez de se distribuírem uniformemente ao longo dos meses —, a análise do comportamento histórico é fundamental para identificar anomalias.

Para ilustrar esse cenário, o gráfico abaixo apresenta a média mensal de precipitação agregada por ano. Como métrica de alerta, estabelecemos um limiar crítico (destacado em vermelho) para os anos em que o volume ficou abaixo de 75% da média histórica. Nota-se que o ano de 2012 situou-se exatamente no limiar de atenção, enquanto 2021 e 2022 registraram baixas pluviométricas severas.

In [ ]:
sazonalidade = df['precipitacao_mm'].groupby(
    df.index.get_level_values('datetime').month
).mean()

print(sazonalidade)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sazonalidade.plot(ax=ax, marker='o', color='steelblue')
ax.set_title('Sazonalidade — Precipitação Média por Mês')
ax.set_ylabel('mm')
ax.set_xlabel('Mês')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun',
                     'Jul','Ago','Set','Out','Nov','Dez'])
ax.axvspan(2, 5, alpha=0.1, color='blue', label='Quadra chuvosa')
ax.legend()
plt.show()

In [ ]:
mascara = df.index.get_level_values('datetime').year.isin([2012, 2016, 2021, 2022])
df_seca = df[mascara]
print(df_seca.shape)

In [ ]:
mascara_normal = ~mascara
df_normal = df[mascara_normal]

In [ ]:
temp_seca = df_seca['temperatura_bulbo_seco_c'].mean()
temp_normal = df_normal['temperatura_bulbo_seco_c'].mean()

print(f"Temperatura média anos de seca: {temp_seca:.2f}°C")
print(f"Temperatura média anos normais: {temp_normal:.2f}°C")

In [ ]:
temp_seca_mensal = df_seca['temperatura_bulbo_seco_c'].groupby(
    df_seca.index.get_level_values('datetime').month
).mean()

temp_normal_mensal = df_normal['temperatura_bulbo_seco_c'].groupby(
    df_normal.index.get_level_values('datetime').month
).mean()

fig, ax = plt.subplots(figsize=(10, 4))
temp_normal_mensal.plot(ax=ax, marker='o', label='Anos normais', color='steelblue')
temp_seca_mensal.plot(ax=ax, marker='o', label='Anos de seca', color='red')

ax.set_title('Temperatura Média por Mês — Seca vs Normal')
ax.set_ylabel('°C')
ax.set_xlabel('Mês')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun',
                     'Jul','Ago','Set','Out','Nov','Dez'])
ax.legend()
plt.show()

### Temperatura em Anos de Seca vs Anos Normais

Anos de seca apresentam temperatura média ligeiramente maior, com a diferença 
mais evidente nos meses fora da quadra chuvosa (junho a setembro).

Durante a quadra chuvosa (fevereiro a maio), a chuva mantém o solo úmido — 
parte da energia solar é consumida pela evapotranspiração, resfriando o ar. 
Nos meses secos de anos de seca, sem água no solo, toda a energia solar vai 
direto para aquecer o ar, elevando a temperatura.

Isso indica que temperatura é uma feature relevante para o modelo — 
ela carrega informação sobre o déficit hídrico, não só sobre o clima.

In [ ]:
diferenca = temp_seca_mensal - temp_normal_mensal
print(diferenca)

In [ ]:
df_estacoes = pd.read_parquet('../data/processed/estacoes.parquet')

fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(df_estacoes['lon'], df_estacoes['lat'],
           color='steelblue', s=100, zorder=5)

for _, row in df_estacoes.iterrows():
    ax.annotate(row['nome'],
                xy=(row['lon'], row['lat']),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=8)

ax.set_title('Estações Meteorológicas — Ceará')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
scale = 12
distribution = climate_indices.indices.Distribution.gamma
data_start_year = 2010
calibration_year_initial = 2010
calibration_year_final = 2024
periodicity = climate_indices.compute.Periodicity.monthly

def calculate_spi(group):
    spi_values = indices.spi(
        group.values,
        scale,
        distribution,
        data_start_year,
        calibration_year_initial,
        calibration_year_final,
        periodicity
    )
    return pd.Series(spi_values, index=group.index)

spi_por_estacao = df['precipitacao_mm'].groupby(
    level='nome_estacao'
).apply(calculate_spi)

print(spi_por_estacao)


In [ ]:
df['spi_12'] = spi_por_estacao.values

fortaleza_spi = df.loc[(slice(None), 'FORTALEZA'), 'spi_12']
fortaleza_spi.index = fortaleza_spi.index.get_level_values('datetime')

fig, ax = plt.subplots(figsize=(14, 4))
fortaleza_spi.plot(ax=ax, color='steelblue', linewidth=0.8)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=-1, color='red', linestyle='--', linewidth=1, label='Seca moderada (SPI < -1)')
ax.fill_between(fortaleza_spi.index, fortaleza_spi, -1,
                where=(fortaleza_spi < -1),
                color='red', alpha=0.3, label='Período de seca')

ax.set_title('SPI-12 — Fortaleza (2010–2024)')
ax.set_ylabel('SPI')
ax.set_xlabel('')
ax.legend()
plt.show()

In [ ]:
quixada_spi = df.loc[(slice(None), 'QUIXADA'), 'spi_12']
quixada_spi.index = quixada_spi.index.get_level_values('datetime')

fig, ax = plt.subplots(figsize=(14, 4))
quixada_spi.plot(ax=ax, color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=-1, color='red', linestyle='--', linewidth=1, label='Seca moderada (SPI < -1)')
ax.fill_between(quixada_spi.index, quixada_spi, -1,
                where=(quixada_spi < -1),
                color='red', alpha=0.3, label='Período de seca')
ax.set_title('SPI-12 — Quixadá (2010–2024)')
ax.set_ylabel('SPI')
ax.legend()
plt.show()

In [ ]:
iguatu_spi = df.loc[(slice(None), 'IGUATU'), 'spi_12']
iguatu_spi.index = iguatu_spi.index.get_level_values('datetime')
iguatu_spi.index.name = None

fig, ax = plt.subplots(figsize=(14, 4))
iguatu_spi.plot(ax=ax, color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=-1, color='red', linestyle='--', linewidth=1, label='Seca moderada (SPI < -1)')
ax.fill_between(iguatu_spi.index, iguatu_spi.values, -1,
                where=(iguatu_spi.values < -1),
                color='red', alpha=0.3, label='Período de seca')
ax.set_title('SPI-12 — Iguatu (2010–2024)')
ax.set_ylabel('SPI')
ax.set_xlabel('')
ax.legend()
plt.show()

In [ ]:
df.to_parquet('../data/processed/climate_mensal.parquet')